<a href="https://colab.research.google.com/github/hiiamlars/einfuehrung_in_die_programmierung_mit_MATLAB/blob/main/open_review_data_copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
# @title Dependencies

# Installations of third-party libraries
!pip install requests -q
!pip install tqdm -q
!pip install beautifulsoup4 -q

# First-party imports
import os
import json
import csv
import random
import argparse
import unicodedata
import threading
import time
import sys
import zipfile
from pathlib import Path
import xml.etree.ElementTree as ET

# Third-party imports
import requests
import pandas as pd
from tqdm.notebook import tqdm
from bs4 import BeautifulSoup
from google.colab import drive

print("\nInstalling and loading dependencies complete!")


Installing and loading dependencies complete!


In [ ]:
# @title Paths

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/iclr/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create RAW_DIR for raw JSONs
RAW_DIR = os.path.join(WORKING_DIR, "raw/")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define and create SAMPLE_DIR for raw and sampled JSONs
SAMPLE_DIR = os.path.join(WORKING_DIR, "sample/")
os.makedirs(SAMPLE_DIR, exist_ok=True)
print(f"Ensured sample directory exists at: {SAMPLE_DIR}.")

# Define and create PDF_DIR for downloaded PDFs
PDF_DIR = os.path.join(WORKING_DIR, "pdf/")
os.makedirs(PDF_DIR, exist_ok=True)
print(f"Ensured PDF directory exists at: {PDF_DIR}.")

# Define and create GROBID_DIR for TEI XML files
GROBID_DIR = os.path.join(WORKING_DIR, "grobid/")
os.makedirs(GROBID_DIR, exist_ok=True)
print(f"Ensured GROBID output directory exists at: {GROBID_DIR}.")

# Define and create TXT_DIR for extracted text files
TXT_DIR = os.path.join(WORKING_DIR, "txt/")
os.makedirs(TXT_DIR, exist_ok=True)
print(f"Ensured TXT output directory exists at: {TXT_DIR}.")

# Define and create LANDINGAI_DIR for LandingAI JSON files
LANDINGAI_DIR = os.path.join(WORKING_DIR, "landingai/")
os.makedirs(LANDINGAI_DIR, exist_ok=True)
print(f"Ensured LandingAI output directory exists at: {LANDINGAI_DIR}.")

# Define and create DATASET_DIR for the resulting data file
DATASET_DIR = os.path.join(WORKING_DIR, "final/")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

print("\nDefining paths complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Thesis/iclr/.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/iclr/raw/.
Ensured sample directory exists at: /content/drive/MyDrive/Thesis/iclr/sample/.
Ensured PDF directory exists at: /content/drive/MyDrive/Thesis/iclr/pdf/.
Ensured GROBID output directory exists at: /content/drive/MyDrive/Thesis/iclr/grobid/.
Ensured TXT output directory exists at: /content/drive/MyDrive/Thesis/iclr/txt/.
Ensured LandingAI output directory exists at: /content/drive/MyDrive/Thesis/iclr/landingai/.
Ensured dataset directory exists at: /content/drive/MyDrive/Thesis/iclr/final/.

Defining paths complete!


In [ ]:
# @title Hardcoded definitions

# Configuration for Sampling
MAX_PAPERS_TO_SAMPLE = 100 # Max number of papers to include in the sample
DATASET_NAME_PREFIX = "rand" # Prefix for the sampled dataset name

# Initialize a list to store failed files while parsing
FAILED_FILENAMES = []

SEED_VALUE = 19260817

print("\nHardcoded definitions complete!")


Hardcoded definitions complete!


In [ ]:
# @title API

# OpenReviewAPI
OPEN_REVIEW_API = "https://api.openreview.net/notes?invitation=ICLR.cc/2023/Conference/-/Blind_Submission&details=directReplies"

# GrobidAPI
GROBID_API = "https://kermitt2-grobid.hf.space/api/processFulltextDocument"

# LandingAIAPI
LANDING_AI_API = 'https://api.va.landing.ai/v1/ade/parse'

## LandingAIAPI-Key
LANDING_AI_API_KEY = "API-Key place holder"

## LandingAI-Header
LANDING_AI_HEADERS = {'Authorization': f'Bearer {LANDING_AI_API_KEY}'}

## LandingAI-Model
LANDING_AI_MODEL = {'model': 'dpt-2-latest'}

print("\nAPI setup complete!")


API setup complete!


# ICLR Data

First, the `get_iclr_reviews()`-function scrapes the paper and review data from the ICLR 2023 dataset.

In [ ]:
def get_iclr_reviews(paper_info_filename=None, review_info_filename=None, decision_info_filename=None):
    """Requests, checks and stores ICLR 2023 reviews via the API."""

    offset = 0
    limit = 100 # Number of paper per request
    max_limit = float('inf') # Maximal amount of requests
    paper_list = [] # Create empty paper list
    review_list = [] # Create empty review list

    while offset + limit <= max_limit:
        print(f'Scraping review data: {offset}/{max_limit}') # Print progress
        print("Start request")
        response = requests.get(OPEN_REVIEW_API, params={"offset": offset, "limit": limit}) # Save responses
        print("End request")
        if response.status_code != 200: # Check for warning messages
            print("Error:", response.status_code)
            break

        data = response.json()
        if not data or 'notes' not in data or not data['notes']: # Check content of response
            break

        for note in data['notes']:
            paper = {} # Create empty storage for the following paper information

            paper['uid'] = note['id']
            paper['number'] = note['number']
            paper['title'] = note['content']['title']
            paper['authors'] = note['content']['authors']
            paper['abstract'] = note['content']['abstract']
            paper['pdf'] = note['content']['pdf']
            paper['keywords'] = note['content']['keywords']

            if 'details' in note and 'directReplies' in note['details']: # Check for complete information
                paper_list.append(paper) # Only then append paper information to the empty list
                for directReplies in note['details']['directReplies']:

                    if directReplies['invitation'].endswith("Official_Review"): # Check for reply information

                        review = {}
                        review['uid'] = directReplies['id']
                        review['paper_uid'] = paper['uid']
                        review['paper_title'] = paper['title']
                        review.update(directReplies['content'])

                        review_list.append(review) # Only then append the paper information to the empty list

        offset += limit # Iterate

    with open(paper_info_filename, 'w', encoding = "utf-8") as f:
        json.dump(paper_list, f) # Write paper_info-file
    with open(review_info_filename, 'w', encoding = "utf-8") as f:
        json.dump(review_list, f) # Write review_info-file

if __name__ == "__main__":
    get_iclr_reviews(paper_info_filename = RAW_DIR + "ICLR2023paper_raw.json", # Create 'paper_info_filename'
                     review_info_filename = RAW_DIR + "ICLR2023review_raw.json") # Create 'review_info_filename'

    print("Finished scraping ICLR 2023 reviews!")

Scraping review data: 0/inf
Start request
End request
Scraping review data: 100/inf
Start request
End request
Scraping review data: 200/inf
Start request
End request
Scraping review data: 300/inf
Start request
End request
Scraping review data: 400/inf
Start request
End request
Scraping review data: 500/inf
Start request
End request
Scraping review data: 600/inf
Start request
End request
Scraping review data: 700/inf
Start request
End request
Scraping review data: 800/inf
Start request
End request
Scraping review data: 900/inf
Start request
End request
Scraping review data: 1000/inf
Start request
End request
Scraping review data: 1100/inf
Start request
End request
Scraping review data: 1200/inf
Start request
End request
Scraping review data: 1300/inf
Start request
End request
Scraping review data: 1400/inf
Start request
End request
Scraping review data: 1500/inf
Start request
End request
Scraping review data: 1600/inf
Start request
End request
Scraping review data: 1700/inf
Start reques

Next, `MAX_PAPERS_TO_SAMPLE` are taken from the whole crawled dataset.

In [ ]:
def to_ascii(input_str):  # Change the utf-8 characters to ascii
    """Converts a string to ASCII, removing non-ASCII characters."""

    assert(type(input_str) == str)
    normalized = unicodedata.normalize("NFKD", input_str)
    normalized = normalized.replace('"', "'")
    ascii_str = "".join(c for c in normalized if c.isascii())
    return ascii_str

if __name__ == "__main__":

    random.seed(SEED_VALUE)
    parser = argparse.ArgumentParser()
    parser.add_argument("--max_count", type=int, default=10**9)
    # Update default input paths to RAW_DIR
    parser.add_argument("--input_paper", type=str, default=os.path.join(RAW_DIR, "ICLR2023paper_raw.json"))
    parser.add_argument("--input_review", type=str, default=os.path.join(RAW_DIR, "ICLR2023review_raw.json"))
    parser.add_argument("--dataset_name", type=str, required=True)

    # Use predefined variables for sampling configuration
    sampled_dataset_name = f"{DATASET_NAME_PREFIX}{MAX_PAPERS_TO_SAMPLE}"
    args = parser.parse_args(["--max_count", str(MAX_PAPERS_TO_SAMPLE), "--dataset_name", sampled_dataset_name])

    # Read businesses in full dataset, and then rename "uid" key to "id"
    with open(args.input_paper, mode="r", encoding="utf-8") as json_file:
        papers = json.load(json_file)
    review_cnt = {}
    for paper in papers:
        paper["id"] = paper["uid"]
        paper.pop("uid", None)
        paper["title"] = to_ascii(str(paper["title"]))
        paper["keywords"] = to_ascii(str(paper["keywords"]))
        paper["abstract"] = to_ascii(str(paper["abstract"]))
        review_cnt[paper["id"]] = 0

    # Read reviews in full dataset
    with open(args.input_review, mode="r", encoding="utf-8") as json_file:
        reviews = json.load(json_file)

    # Create review format string
    review_format_str = '''Summary Of The Paper:

{}

Strength And Weaknesses:

{}

Clarity, Quality, Novelty And Reproducibility:

{}

Summary Of The Review:

{}
'''

    for review in reviews:
        review["belong_id"] = review["paper_uid"]
        review.pop("paper_uid", None)
        review["review"] = review_format_str.format(review["summary_of_the_paper"],review["strength_and_weaknesses"],review["clarity,_quality,_novelty_and_reproducibility"],review["summary_of_the_review"])
        review_cnt[review["belong_id"]] += 1

    paper_selected = []
    has = {}
    for paper in papers:

        def legal(paper):
            if review_cnt[paper["id"]] < 3: return False
            return True

        if legal(paper):

            paper_selected.append(paper)
            has[paper["id"]] = []

    # Randomizes and limits the amount of papers selected
    random.shuffle(paper_selected)
    if len(paper_selected) > args.max_count:
        paper_selected = paper_selected[:args.max_count]

    print("# items =", len(paper_selected))
    with open(os.path.join(SAMPLE_DIR, f"paper_{args.dataset_name}.json"),"w",encoding="utf-8") as json_file:
        json.dump(paper_selected, json_file)

    for i, review in enumerate(reviews):
        if review["belong_id"] in has:
            has[review["belong_id"]].append(i)
    review_selected = []
    for paper in paper_selected:
        for i in has[paper["id"]]:
            reviews[i]["review"] = to_ascii(reviews[i]["review"])
            review_selected.append(reviews[i])

    print("# reviews =", len(review_selected))
    with open(os.path.join(SAMPLE_DIR, f"review_{args.dataset_name}.json"),"w",encoding="utf-8") as json_file:
        json.dump(review_selected, json_file)

    print("\nSampling complete!")

# items = 100
# reviews = 373

Sampling complete!


# Original PDFs


For every paper in the final sample the original PDF will be downloaded as well.

In [ ]:
# Open sampled paper info
with open(os.path.join(SAMPLE_DIR, f"paper_{args.dataset_name}.json"), "r") as json_file:
    papers = json.load(json_file)

# Open sampled review info
with open(os.path.join(SAMPLE_DIR, f"review_{args.dataset_name}.json"), "r") as json_file:
    reviews = json.load(json_file)

def is_file_larger_than(path,lim): # 10 kb
    """Checks if a file is larger than a given limit in bytes."""

    try:
        size = os.path.getsize(path)
        return size > lim
    except FileNotFoundError:
        return False

if __name__ == "__main__":

    # Show download progress
    for i, paper in enumerate(tqdm(papers, desc="Downloading Papers")):

        # Create path for downloaded PDFs using PDF_DIR
        path_pdf = os.path.join(PDF_DIR, paper["id"] + ".pdf")


        if not is_file_larger_than(path_pdf, 10*1024):
            time.sleep(0.5)
            url = "https://openreview.net" + paper["pdf"]
            print(i, url)
            response = requests.get(url, proxies={"http": None, "https": None})

            # Ensure success
            if response.status_code == 200:
                with open(path_pdf, "wb") as file:
                    file.write(response.content)

            # In case of failure
            else:
                print("Failed to download the file. Status code:", response.status_code)
                time.sleep(2)
                response = requests.get(url, proxies={"http": None, "https": None})
                if response.status_code == 200:
                    with open(path_pdf, "wb") as file:
                        file.write(response.content)
                else:
                    print("Failed again:", response.status_code)
                    raise(Exception)

    print("\nDownload complete!")

0 https://openreview.net/pdf/b7b54047fdaf97f505713a0b6675abc7e120c460.pdf
1 https://openreview.net/pdf/e0646a302829ac0b05b63973439d7651735a4498.pdf
2 https://openreview.net/pdf/20516df845666b5c362ac8bd1c616c7eb44ad606.pdf
3 https://openreview.net/pdf/043fba8d0ed8251ba2eb757665721e7fc496d839.pdf
4 https://openreview.net/pdf/106dc37f05b949150cc78b90e79512c41069b3a9.pdf
5 https://openreview.net/pdf/e48b4b7a07d7810a1f1175bb20762f88e7436ae8.pdf
6 https://openreview.net/pdf/98c0a5bad8225b6d1baf5c74047c4d04bacfcfa1.pdf
7 https://openreview.net/pdf/7cc43ee917a76ab228da527b72fbf90f70fb6608.pdf
8 https://openreview.net/pdf/375cd8060d66d3802592086cf8d856f477c86f39.pdf
9 https://openreview.net/pdf/49e74635805c5667e19c63ecaab734e6a4c18e0c.pdf
10 https://openreview.net/pdf/64363982c84b5f6eb48debf9bb39680c33562378.pdf
11 https://openreview.net/pdf/326a8562c6392857dcaaa121dcbbaaf8902b235f.pdf
12 https://openreview.net/pdf/ad0c6ee1a1c026a3dec4173db20dede07c063f3c.pdf
13 https://openreview.net/pdf/615dc

The PDFs are zipped up into a single folder.

In [ ]:
if __name__ == "__main__":

    # Define locations
    path_folder = PDF_DIR
    path_zip = os.path.join(path_folder, "paper.zip")

    file_paths = []
    # Iterate through the directory
    for root, _, files in os.walk(path_folder):
        for file in files:
            if file.endswith(".pdf"):
                file_paths.append(os.path.join(root, file))

    # In case no PDFs are present
    if not file_paths:
        print(f"No PDF files found in {path_folder} to zip.")
    # Else place the PDF(s) in the ZIP file
    else:
        with zipfile.ZipFile(path_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for file in tqdm(file_paths, desc="Zipping files"):
                zipf.write(file, arcname=os.path.basename(file))

        print("\nZIP-file created!")

Zipping files:   0%|          | 0/100 [00:00<?, ?it/s]


ZIP-file created!


# Parsing

Now that the PDFs are downloaded, they can be parsed using the GROBID-API.

In [ ]:
# Define input path (PDF-files) using PDF_DIR
INPUT_DIR = PDF_DIR

# Define output path (TEI XML-files) using GROBID_DIR
OUTPUT_DIR = GROBID_DIR

# Define API-call function
def pdf2grobid(filename, GROBID_API):
    """Sends a single PDF to the GROBID API and returns the TEI XML text."""

    try:
        with open(filename, 'rb') as file:
            files = {'input': file}
            # Long timeout for safety
            response = requests.post(GROBID_API, files=files, timeout=300)

        # Check for response error
        if response.status_code != 200:
            print(f"API Error {response.status_code}: {response.text}")
            response.raise_for_status()

        # Capture the text of the API response (expected as tei_xml)
        return response.text

    # Handle errors
    except requests.exceptions.RequestException as e:
        print(f"Error processing {filename}: {e}")
        return None

if __name__ == "__main__":
    # Get list of PDF files to process
    pdf_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".pdf")]

    # Iterate over all PDF files in the input directory
    for filename in tqdm(pdf_files, desc="Processing PDFs with GROBID"):
        # Construct the full input path
        input_file_path = os.path.join(INPUT_DIR, filename)

        # Replace .pdf with .xml for the output filename
        xml_filename = filename.replace(".pdf", ".xml")

        # Construct the full output path
        output_file_path = os.path.join(OUTPUT_DIR, xml_filename)

        # print(f"Processing: {filename}...")

        # Call defined function for parsing
        tei_xml = pdf2grobid(input_file_path, GROBID_API)

        if tei_xml:
            # Write a TEI XML-file into the output path
            with open(output_file_path, 'w', encoding='utf-8') as f:
                f.write(tei_xml)
            # print(f"  -> Success: Saved to {xml_filename}")

        else:
            # print(f"  -> Failure: Could not process {filename}.")
            FAILED_FILENAMES.append(filename)

        # Add short delay for safety
        time.sleep(3)

    print("\nProcessing complete!")
    print(f"Failed filenames: {FAILED_FILENAMES}")

Processing: vuD2xEtxZcj.pdf...
  -> Success: Saved to vuD2xEtxZcj.xml
Processing: WoByU5W5te0.pdf...
  -> Success: Saved to WoByU5W5te0.xml
Processing: XZRmNjUMj0c.pdf...
  -> Success: Saved to XZRmNjUMj0c.xml
Processing: cDYRS5iZ16f.pdf...
  -> Success: Saved to cDYRS5iZ16f.xml
Processing: Sh97TNO5YY_.pdf...
  -> Success: Saved to Sh97TNO5YY_.xml
Processing: QVcDQJdFTG.pdf...
  -> Success: Saved to QVcDQJdFTG.xml
Processing: ju_Uqw384Oq.pdf...
  -> Success: Saved to ju_Uqw384Oq.xml
Processing: Z8qk2iM5uLI.pdf...
API Error 503: <html>
<head>
<meta http-equiv="Content-Type" content="text/html;charset=ISO-8859-1"/>
<title>Error 503 Service Unavailable</title>
</head>
<body><h2>HTTP ERROR 503 Service Unavailable</h2>
<table>
<tr><th>URI:</th><td>/api/processFulltextDocument</td></tr>
<tr><th>STATUS:</th><td>503</td></tr>
<tr><th>MESSAGE:</th><td>Service Unavailable</td></tr>
<tr><th>SERVLET:</th><td>jersey</td></tr>
</table>

</body>
</html>

Error processing /content/drive/MyDrive/Thesis

Some papers were not able to be parsed via the GROBID-API. Xu et al. (2024) also reported this problem. Therefore another attempt was made with the landingAI-API.

In [ ]:
# Define input path (PDF-files) using PDF_DIR
INPUT_DIR = PDF_DIR

# Define output dir (JSON-files) using LANDINGAI_DIR
OUTPUT_DIR_JSON = LANDINGAI_DIR

# Define the new function for the LandingAI API call
def pdf_to_landingai_json(filename, url, headers, data):
    """Sends a single PDF to the LandingAI API and returns the parsed JSON."""

    try:
        # Open the PDF in binary mode for upload
        with open(filename, 'rb') as document:
            files = {'document': document}

            # Send the request
            # Use timout for safety
            response = requests.post(url, files=files, data=data, headers=headers, timeout=300)

        # Check for response error
        if response.status_code != 200:
            print(f"\n  -> API Error {response.status_code}: {response.text}")
            response.raise_for_status()

        # Capture response as json
        return response.json()

    # In case of an error
    except requests.exceptions.RequestException as e:
        print(f"\n  -> Connection Error: {e}")
        return None

if __name__ == "__main__":
    # Print start of the process
    print(f"Starting retry process for {len(FAILED_FILENAMES)} files using LandingAI...")

    for filename in FAILED_FILENAMES:

        # Construct the full input path
        input_file_path = os.path.join(INPUT_DIR, filename)

        # Construct the full output path
        json_filename = filename.replace(".pdf", ".json")
        output_file_path = os.path.join(OUTPUT_DIR_JSON, json_filename)

        # Print current status
        print(f"Processing: {filename}...")

        # Call the new function
        parsed_json_output = pdf_to_landingai_json(
            filename=input_file_path, url=LANDING_AI_API, headers=LANDING_AI_HEADERS, data=LANDING_AI_MODEL
        )

        if parsed_json_output:
            # Write json-file
            with open(output_file_path, 'w', encoding='utf-8') as f:
                # Use indent=4 for readable formatting
                json.dump(parsed_json_output, f, indent=4)

            # Message in case of success
            print(f"  -> Success: Saved JSON to {json_filename}")
        else:
            # Message in case of failure
            print(f"  -> Final failure: Could not process {filename}.")

        # Add short delay for safety
        time.sleep(1)

    print("\nProcessing complete!")

Starting retry process for 2 files using LandingAI...
Processing: Z8qk2iM5uLI.pdf...

  -> API Error 401: {"error":"Invalid Authorization header format"}

  -> Connection Error: 401 Client Error: Unauthorized for url: https://api.va.landing.ai/v1/ade/parse
  -> Final failure: Could not process Z8qk2iM5uLI.pdf.
Processing: lrzX-rNuRvw.pdf...

  -> API Error 401: {"error":"Invalid Authorization header format"}

  -> Connection Error: 401 Client Error: Unauthorized for url: https://api.va.landing.ai/v1/ade/parse
  -> Final failure: Could not process lrzX-rNuRvw.pdf.

Processing complete!


However, the same papers PDFs also failed to be converted by the landingAI-API.

# Conversion

The PDFs who were successfully parsed by the GROBID-API are next converted to the .txt-format.

In [ ]:
# Define input directory (where GROBID XML files are stored) using GROBID_DIR
INPUT_DIR = GROBID_DIR

# Define output directory for TXT files using TXT_DIR
OUTPUT_DIR = TXT_DIR
# Create the output path if not already existing (already done in Paths cell)

# Define extraction function
def extract_text_from_tei_xml(xml_content):
    """Parses TEI XML and extracts clean body text."""

    # Use 'lxml' as the parser for speed and robustness with XML
    soup = BeautifulSoup(xml_content, 'lxml')

    body_tag = soup.find('body') # 'body' = main text

    if body_tag:
        # .get_text(separator=' ', strip=True) extracts all nested text
        # And cleanly separates it with spaces, removing excess newlines/whitespace.
        clean_text = body_tag.get_text(separator=' ', strip=True)
        return clean_text
    return "" # Return empty string if body tag is not found

if __name__ == "__main__":
    # Iterate across all .xml-files in the INPUT_XMR_DIR
    xml_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.xml')]
    # Message how many files to convert
    print(f"Found {len(xml_files)} XML files to convert.")

    # Message progress of file conversion
    for filename in tqdm(xml_files, desc="Converting XML to TXT"):

        # Construct the full input path
        xml_path = os.path.join(INPUT_DIR, filename)

        # Replace .xml with .txt for the output file name
        txt_filename = filename.replace(".xml", ".txt")

        # Construct the full output path
        txt_path = os.path.join(OUTPUT_DIR, txt_filename)

        try:
            # Read the input fill
            with open(xml_path, 'r', encoding='utf-8') as f:
                xml_content = f.read()

            # Call the defined function for extraction
            parsed_text = extract_text_from_tei_xml(xml_content)

            # Write a TXT-file into the output path
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(parsed_text)

        # Print error message
        except Exception as e:
            print(f"\nError processing {filename}: {e}")
            continue # Skip to the next file

    print("\nConversion complete! Text files are now in the 'txt' folder.")

Found 98 XML files to convert.


Converting XML to TXT:   0%|          | 0/98 [00:00<?, ?it/s]

/tmp/ipython-input-3726282015.py:13: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(xml_content, 'lxml')



Conversion complete! Text files are now in the 'txt' folder.


# Final dataset

The metadata of the paper, the human reviews and the parsed text are all combined into a final dataframe `dataset_paper.parquet`.

In [ ]:
if __name__ == "__main__":
    # Open random sampled paper and review data
    with open(os.path.join(SAMPLE_DIR, f"paper_{args.dataset_name}.json"), "r") as json_file:
        papers = json.load(json_file)
    with open(os.path.join(SAMPLE_DIR, f"review_{args.dataset_name}.json"), "r") as json_file:
        reviews = json.load(json_file)

    # Iterates through all reviews and maps all of them onto their ID
    mp = {}
    for review in reviews:
        if review["belong_id"] not in mp:
            mp[review["belong_id"]] = []
        mp[review["belong_id"]].append(review["review"])

    # Ensurs sufficient and random picked reviews for each paper
    assert(len(mp)==len(papers))
    cnt = 0
    for (paper_id, reviews_list) in mp.items():
        assert(len(reviews_list) >= 3)
        random.seed(SEED_VALUE + cnt)
        cnt += 1
        random.shuffle(reviews_list)

    # Create dataframe
    df = pd.DataFrame(columns=["paper_id", "pdf_url", "abstract", "parsed_text","human_review_1","human_review_2","human_review_3"]) # With respective columns
    for i, paper in enumerate(tqdm(papers, desc="Processing Papers")):
        pdf_url = "https://openreview.net" + paper["pdf"]

        # If a txt file is present it gets attached
        try:
            txt_filepath = os.path.join(TXT_DIR, paper["id"] + ".txt")
            with open(txt_filepath, "r", encoding="utf-8") as f:
                parsed_text = f.read()

        # If the file is missing the paper is skipped
        except FileNotFoundError:
            print(f"Skipping paper (Raw text file not found at: {txt_filepath}).")
            continue
        except Exception as e:
            print(f"Error reading raw text for paper {paper['id']}: {e}")
            continue

        # Place the respective content
        df.loc[len(df)] = [paper["id"], pdf_url, paper["abstract"], parsed_text, mp[paper["id"]][0],mp[paper["id"]][1],mp[paper["id"]][2]]

    # Save dataset as .parquet
    df.to_parquet(os.path.join(DATASET_DIR, 'dataset_paper.parquet'), index=False)

    print("\nDataset created!")

Processing Papers:   0%|          | 0/100 [00:00<?, ?it/s]

Skipping paper (Raw text file not found at: /content/drive/MyDrive/Thesis/iclr/txt/Z8qk2iM5uLI.txt).
Skipping paper (Raw text file not found at: /content/drive/MyDrive/Thesis/iclr/txt/lrzX-rNuRvw.txt).

Dataset created!


The just created `dataset_paper.parquet` is loaded, checked for odd entries and visually inspected.

In [ ]:
if __name__ == "__main__":
    # Read dataset_paper
    dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "dataset_paper.parquet"))

    for i in range(len(dataset_paper)):
            row = dataset_paper.loc[i] # Retrive the i-th row
            if len(row["parsed_text"]) < 1000: # Check if length "parsed_text" < 1000 (untypical short)
                print(i, row["paper_id"], len(row["parsed_text"])) # Print index, paper ID and length of "parsed_text"

    # Print head
    display(dataset_paper.head(10))

    print("\nDataset head printed!")

,paper_id,pdf_url,abstract,parsed_text,human_review_1,human_review_2,human_review_3
0,vuD2xEtxZcj,https://openreview.net/pdf/b7b54047fdaf97f5057...,"In deep learning, fine-grained N:M sparsity re...",MINIMUM VARIANCE UNBIASED N:M SPARSITY FOR THE...,Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...
1,WoByU5W5te0,https://openreview.net/pdf/e0646a302829ac0b05b...,We present a novel method to regularizes neura...,GECONERF: FEW-SHOT NEURAL RADIANCE FIELDS VIA ...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focus on t...,Summary Of The Paper:\n\nThe paper proposes a ...
2,XZRmNjUMj0c,https://openreview.net/pdf/20516df845666b5c362...,Neural Architecture Search (NAS) is a fast-dev...,EFFICIENT ONE-SHOT NEURAL ARCHITECTURE SEARCH ...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focused on...,Summary Of The Paper:\n\nThe paper introduces ...
3,cDYRS5iZ16f,https://openreview.net/pdf/043fba8d0ed8251ba2e...,Scaling transformers has led to significant br...,LEARNING TO GROW PRETRAINED MODELS FOR EFFICIE...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThe paper proposes a ...,Summary Of The Paper:\n\nThis paper investigat...
4,Sh97TNO5YY_,https://openreview.net/pdf/106dc37f05b949150cc...,We are interested in in silico evaluation meth...,BIASES IN EVALUATION OF MOLECULAR OPTIMIZA-TIO...,Summary Of The Paper:\n\nThis paper analyses r...,Summary Of The Paper:\n\nThe authors propose a...,"Summary Of The Paper:\n\nIn this work, the aut..."
5,QVcDQJdFTG,https://openreview.net/pdf/e48b4b7a07d7810a1f1...,We propose preventive learning as the first fr...,ENSURING DNN SOLUTION FEASIBILITY FOR OPTI-MIZ...,Summary Of The Paper:\n\nThe paper presents an...,Summary Of The Paper:\n\nThis paper proposes t...,Summary Of The Paper:\n\nThis paper proposes a...
6,ju_Uqw384Oq,https://openreview.net/pdf/98c0a5bad8225b6d1ba...,Time series analysis is of immense importance ...,TIMESNET: TEMPORAL 2D-VARIATION MODELING FOR G...,Summary Of The Paper:\n\n\nThis paper focuses ...,Summary Of The Paper:\n\nThis paper focuses on...,Summary Of The Paper:\n\nThe authors argue tha...
7,yHIIM9BgOo,https://openreview.net/pdf/375cd8060d66d380259...,We propose an actor-critic framework for graph...,GRAPH-BASED DETERMINISTIC POLICY GRADIENT FOR ...,Summary Of The Paper:\n\nThe paper proposes an...,Summary Of The Paper:\n\nThis paper introduces...,Summary Of The Paper:\n\nThis paper proposes a...
8,deit1AdsFU,https://openreview.net/pdf/64363982c84b5f6eb48...,Gradient inversion attack enables recovery of ...,LEARNING TO INVERT: SIMPLE ADAPTIVE ATTACKS FO...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThe paper introduces ...,Summary Of The Paper:\n\nThis paper investigat...
9,pmUH7A8wZz,https://openreview.net/pdf/326a8562c6392857dca...,With the recent advance of geometric deep lear...,AUTOENCODING HYPERBOLIC REPRESENTATION FOR ADV...,Summary Of The Paper:\n\nA proposal of hyperbo...,Summary Of The Paper:\n\nThe paper presents a ...,Summary Of The Paper:\n\nThis work proposes a ...



Dataset head printed!


# References

The main code logic and in parts exact code has been taken from:

Xu, S., Lu, Y., Schoenebeck, G., & Kong, Y. (2024). Benchmarking LLMs'   Judgments with No Gold Standard. *arXiv preprint arXiv:2411.07127.*

The repository is available [here](https://github.com/yx-lu/Benchmarking-LLMs--Judgments-with-No-Gold-Standard)
